In [22]:
import boto3

# Assume into OrganizationAccountAccessRole in a target account
# By default this uses your *current* account ID; change TARGET_ACCOUNT_ID
# if you want to assume into a different AWS account.
base_session = boto3.session.Session()
sts = base_session.client("sts")
current_identity = sts.get_caller_identity()

TARGET_ACCOUNT_ID = 396511695522  # TODO: set to another account ID if needed
ROLE_NAME = "OrganizationAccountAccessRole"
ROLE_ARN = f"arn:aws:iam::{TARGET_ACCOUNT_ID}:role/{ROLE_NAME}"

print("Assuming role:", ROLE_ARN)
assumed = sts.assume_role(
    RoleArn=ROLE_ARN,
    RoleSessionName="rgasa-purge-session",
)
creds = assumed["Credentials"]

# Create a session with the assumed-role credentials
assumed_session = boto3.session.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=base_session.region_name or "eu-west-1",
)

# Optionally make the assumed role the default for future boto3.client/resource calls
boto3.setup_default_session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=base_session.region_name or "eu-west-1",
)

# Confirm who we are now
identity = assumed_session.client("sts").get_caller_identity()

print("AWS STS get_caller_identity() after assume-role:")
print(f"  Account:   {identity['Account']}")
print(f"  UserId:    {identity['UserId']}")
print(f"  ARN:       {identity['Arn']}")
print(f"  Region:    {assumed_session.region_name}")

print("You can now use boto3.client(...) and it will use the assumed role.")

Assuming role: arn:aws:iam::396511695522:role/OrganizationAccountAccessRole
AWS STS get_caller_identity() after assume-role:
  Account:   396511695522
  UserId:    AROAJ5YTGMWHKRNBYDR4Y:rgasa-purge-session
  ARN:       arn:aws:sts::396511695522:assumed-role/OrganizationAccountAccessRole/rgasa-purge-session
  Region:    eu-west-1
You can now use boto3.client(...) and it will use the assumed role.


In [23]:
import boto3

logs_client = boto3.client("logs")

# List log groups so we can find the exact name
paginator = logs_client.get_paginator("describe_log_groups")

matching = []
for page in paginator.paginate():
    for lg in page.get("logGroups", []):
        name = lg.get("logGroupName")
        if "comodash" in name.lower():
            matching.append(name)

print("Log groups containing 'comodash':")
for name in matching:
    print(" -", name)

if not matching:
    print("No log groups containing 'comodash' were found. Check AWS console for the exact log group name.")

Log groups containing 'comodash':
 - /api/comodash/
 - /aws/codebuild/ComoDashAPIv2-CodeBuild


In [ ]:
import boto3
import json
from datetime import datetime, timezone

# Fetch all CloudWatch Logs events for /api/comodash/
# between 7 Dec 2025 (inclusive) and 10 Dec 2025 (inclusive).
# Assumes you have already run the assume-role cell above.

LOG_GROUP_NAME = "/api/comodash/"
start_day=5
end_day=9
end_month=12

start_dt = datetime(2025, 12, start_day, 0, 0, 0, tzinfo=timezone.utc)
end_dt = datetime(2025, 12, end_day, 23, 59, 59, tzinfo=timezone.utc)  # end exclusive; includes 7–10 Dec

start_ms = int(start_dt.timestamp() * 1000)
end_ms = int(end_dt.timestamp() * 1000)

output_path = f"comodash_api_logs_2025-12-{start_day}_to_2025-12-{end_day}.jsonl"

logs_client = boto3.client("logs")  # will use the assumed role if you ran the assume-role cell

all_events = []
next_token = None
page = 0

while True:
    params = {
        "logGroupName": LOG_GROUP_NAME,
        "startTime": start_ms,
        "endTime": end_ms,
        "limit": 10000,
    }
    if next_token is not None:
        params["nextToken"] = next_token

    response = logs_client.filter_log_events(**params)
    page += 1

    events = response.get("events", [])
    all_events.extend(events)
    print(f"Fetched page {page}, {len(events)} events (total so far: {len(all_events)})")

    next_token = response.get("nextToken")
    if not next_token:
        break

with open(output_path, "w", encoding="utf-8") as f:
    for event in all_events:
        f.write(json.dumps(event, ensure_ascii=False))
        f.write("\n")

print(f"Downloaded {len(all_events)} events from '{LOG_GROUP_NAME}' between {start_dt} and {end_dt} into {output_path}.")

Fetched page 1, 921 events (total so far: 921)
Fetched page 2, 1243 events (total so far: 2164)


In [25]:
import json
from collections import defaultdict


# logStreamName -> list of events
streams = defaultdict(list)

with open(output_path) as f:
    for line in f:
        if not line.strip():
            continue
        outer = json.loads(line)
        inner = json.loads(outer["message"])

        # Skip the /identity/security_metadata route
        if inner.get("routeKey") == "GET /identity/security_metadata":
            continue

        streams[outer["logStreamName"]].append({
            "timestamp": outer["timestamp"],
            **inner,
        })

# Optional: sort events in each stream by timestamp
for stream, events in streams.items():
    events.sort(key=lambda e: e["timestamp"])

# Show the events for each stream
for stream, events in streams.items():
    print(f"\n=== stream: {stream} ({len(events)} events) ===")
    for e in events:
        print(e)


=== stream: kgh5ifwnjc_.default-2025-12-05-07-10 (1 events) ===
{'timestamp': 1764918654507, 'requestId': 'VGnL1jZ0DoEEJRA=', 'ip': '52.17.153.158', 'requestTime': '05/Dec/2025:07:10:54 +0000', 'httpMethod': 'GET', 'routeKey': 'GET /query/credentials', 'status': '200', 'protocol': 'HTTP/1.1', 'responseLength': '1390'}

=== stream: kgh5ifwnjc_.default-2025-12-05-07-11 (1 events) ===
{'timestamp': 1764918662314, 'requestId': 'VGnNDiSxjoEEPVQ=', 'ip': '52.17.153.158', 'requestTime': '05/Dec/2025:07:11:02 +0000', 'httpMethod': 'GET', 'routeKey': 'GET /query/credentials', 'status': '200', 'protocol': 'HTTP/1.1', 'responseLength': '1390'}

=== stream: kgh5ifwnjc_.default-2025-12-05-07-12 (1 events) ===
{'timestamp': 1764918722368, 'requestId': 'VGnWbgOFjoEEPOQ=', 'ip': '52.17.153.158', 'requestTime': '05/Dec/2025:07:12:02 +0000', 'httpMethod': 'GET', 'routeKey': 'GET /query/credentials', 'status': '200', 'protocol': 'HTTP/1.1', 'responseLength': '1390'}

=== stream: kgh5ifwnjc_.default-2025

In [12]:
target_request_id = "U5hviiNWDoEEPFg="

# Flatten all events and filter by requestId
matching = [
    (stream, e)
    for stream, events in streams.items()
    for e in events
    if e.get("requestId") == target_request_id
]

print(f"Found {len(matching)} events for requestId={target_request_id}")
for stream, e in matching:
    print(f"\nstream={stream}")
    print(e)

Found 1 events for requestId=U5hviiNWDoEEPFg=

stream=kgh5ifwnjc_.default-2025-12-01-07-53
{'timestamp': 1764575638238, 'requestId': 'U5hviiNWDoEEPFg=', 'ip': '52.17.153.158', 'requestTime': '01/Dec/2025:07:53:58 +0000', 'httpMethod': 'GET', 'routeKey': 'GET /query/credentials', 'status': '200', 'protocol': 'HTTP/1.1', 'responseLength': '1378'}
